# Octo resident adapter (`--backend octo`)

Serves Octo (JAX/Flax) from its own resident Flask adapter, `control/vla_bridge/octo_server.py`, through an ngrok HTTP tunnel.

| | |
|---|---|
| Lane | HTTP adapter (isolated Python 3.10 venv, JAX) |
| Port → tunnel | 3004 → ngrok HTTP |
| GPU runtime | L4 |
| Gated HF licences | none |
| Sign-offs before running | 14-UAT 5 licence decision |
| Default checkpoint | `sapanostic/octo-so101-3hz-finetune@a45ff970` |
| Server log | `/content/octo_server.log` |
| Embedded files (written to `/content`) | `colab_server_utils.py`, `octo_server.py` |

Run top to bottom on a fresh runtime. Works from the VS Code Colab extension: nothing to upload.
Every file this notebook runs is written into `/content` by a cell below and checked against
its sha256. Run the teardown cell at the end of every session. The index of all notebooks,
with the local smoke and episode commands, is `control/vla_bridge/notebooks/README.md`.

Generated by `scripts/build_colab_notebooks.py` from `scripts/colab_notebook_specs.py` +
`control/vla_bridge` sources, so edit those and regenerate, not this file.

In [ ]:
import os

# Test 7 UAT crash investigation (Phase 12, 2026-09-27): Colab sessions that
# load multiple ML libraries (torch, opencv-headless, Fast-FoundationStereo's
# own deps) commonly end up with more than one bundled OpenMP runtime
# (libgomp vs libiomp) loaded into the same process -- the classic
# "OMP: Error #15" failure mode, which manifests as an abrupt, tracebackless
# kernel death the first time a *new* library's compute path runs for the
# first time (matches "crashes right as inference starts"). Set before any
# heavy imports so it applies process-wide to every cell below.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
print("OpenMP duplicate-runtime guards set.")

**pyngrok legitimacy gate:** the next cell installs `pyngrok`, flagged
`SUS` in `11-RESEARCH.md`'s Package Legitimacy Audit. Do **not** run that
install cell until a human has completed the verification in this plan's
Task 3 `checkpoint:human-verify` (visit pypi.org/project/pyngrok, confirm
the `ngrok`/`alexdlaird` maintainer namespace and that `8.1.2` is a real
published version).

In [ ]:
# Colab notebook cell — DO NOT RUN until the human has completed the
# pyngrok legitimacy check in this plan's Task 3 checkpoint:human-verify
!pip install pyngrok==8.1.2

In [ ]:
# One-time token setup: HF Hub (for authenticated, faster/rate-limit-free checkpoint
# downloads) + ngrok (for the TCP tunnel). Checks for saved token files on Google Drive
# first (same MyDrive/SoARM-Research/.hf_token / .ngrok_token convention as other
# cells in this project), else falls back to a one-time getpass() prompt so tokens
# are never echoed or left in cell output/notebook history.
import getpass
from pathlib import Path

_hf_token = None
_ngrok_token = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _hf_token_path = Path("/content/drive/MyDrive/SoARM-Research/.hf_token")
    _ngrok_token_path = Path("/content/drive/MyDrive/SoARM-Research/.ngrok_token")
    if _hf_token_path.exists():
        _hf_token = _hf_token_path.read_text().strip()
        print(f"HF token loaded from {_hf_token_path}")
    if _ngrok_token_path.exists():
        _ngrok_token = _ngrok_token_path.read_text().strip()
        print(f"ngrok token loaded from {_ngrok_token_path}")
except Exception as _exc:
    print(f"Drive mount/token read skipped ({type(_exc).__name__}: {_exc})")

if not _hf_token:
    _hf_token = getpass.getpass("HF Hub token (paste here): ")
if not _ngrok_token:
    _ngrok_token = getpass.getpass("ngrok authtoken (paste here): ")

from huggingface_hub import login
login(token=_hf_token)
print("Logged into HF Hub.")

from pyngrok import ngrok
ngrok.set_auth_token(_ngrok_token)
print("ngrok authtoken set.")

del _hf_token, _ngrok_token

## Server helper

The next cell writes `control/vla_bridge/colab_server_utils.py` into `/content/colab_server_utils.py`; the cell after it checks its sha256. The launch, diagnostic and teardown cells import it to stop stale servers, prove this notebook's server owns its port, and report logs, processes and GPU memory.

In [ ]:
%%writefile /content/colab_server_utils.py
"""Colab-side process/port helper for the per-backend policy-server notebooks.

Every notebook in control/vla_bridge/notebooks/ writes this file to
/content/colab_server_utils.py with a %%writefile cell (byte-identical to this file, checked
by a sha256 cell and by control/test_colab_notebooks.py) and imports it with
sys.path.insert(0, '/content'). Local control code never imports it.

What it guarantees for a launch cell:
- stop_servers() clears any earlier server: every process whose cmdline contains a known
  server marker, plus whatever is listening on the notebook's port.
- start_server() starts the server unbuffered (PYTHONUNBUFFERED=1) with its log in a file.
- wait_until_ready() only succeeds when the process this cell started (or one of its
  children) owns the port. A stale server that answers on the port is an error, not a
  success. It also fails on an early exit and, for patched launchers, on a missing banner.

The diagnostic helpers (log_report, process_report, gpu_report, health_report) print what a
partner needs to debug a run, and stop_server() is the teardown.

Imports: stdlib + psutil only (psutil is preinstalled on Colab).
"""

import json
import os
import re
import socket
import subprocess
import time
import urllib.request
from pathlib import Path
from typing import Callable, Mapping, Optional, Sequence

import psutil

SERVER_MARKERS = (
    "lerobot.async_inference.policy_server",
    "diffusion_policy_server.py",
    "groot_n17_server.py",
    "fastfs_server.py",
    "openvla_oft_server.py",
    "openvla_server.py",
    "octo_server.py",
    "wallx_server.py",
    "rdt1b_server.py",
    "groot_n15_server.py",
)

# Case-sensitive on purpose: "Error" covers KeyError, NotImplementedError, GatedRepoError and
# "Error in StreamActions"; lowercase prose such as "error budget" is not a match.
ERROR_PATTERN = re.compile(
    r"ERROR|Error|Traceback|Exception|OutOfMemory|out of memory|Killed|beginAllocateToPool"
)

_ACCESS_ERRORS = (psutil.AccessDenied, psutil.NoSuchProcess, psutil.ZombieProcess)


def _cmdline(proc) -> str:
    try:
        return " ".join(proc.cmdline())
    except _ACCESS_ERRORS:
        return ""
    except OSError:
        return ""


def _describe(pid: int) -> str:
    try:
        return f"PID {pid} ({_cmdline(psutil.Process(pid)) or '?'})"
    except _ACCESS_ERRORS:
        return f"PID {pid} (gone)"


def _tcp_connections(proc):
    # Colab's psutil may predate 6.0, where net_connections() replaced connections().
    method = getattr(proc, "net_connections", None) or proc.connections
    return method(kind="tcp")


def _read_lines(log_path) -> list:
    path = Path(log_path)
    if not path.exists():
        return []
    return path.read_text(errors="replace").splitlines()


def _tail(log_path, lines: int = 40) -> str:
    return "\n".join(_read_lines(log_path)[-lines:])


def listening_pids(port: int) -> set:
    """PIDs of every visible process with a TCP socket in LISTEN state on `port`."""
    pids = set()
    for proc in psutil.process_iter():
        try:
            for conn in _tcp_connections(proc):
                if conn.status == psutil.CONN_LISTEN and conn.laddr and conn.laddr.port == port:
                    pids.add(proc.pid)
        except _ACCESS_ERRORS:
            continue
        except OSError:
            continue
    return pids


def stop_servers(port: int, patterns: Sequence[str] = (), timeout_s: float = 15.0) -> list:
    """Stop every process matching a pattern plus the port's listener; return what was stopped."""
    me = os.getpid()
    targets = {}
    for proc in psutil.process_iter():
        if proc.pid == me:
            continue
        cmdline = _cmdline(proc)
        if cmdline and any(pattern and pattern in cmdline for pattern in patterns):
            targets[proc.pid] = proc
    for pid in listening_pids(port):
        if pid != me and pid not in targets:
            try:
                targets[pid] = psutil.Process(pid)
            except _ACCESS_ERRORS:
                continue
    stopped = []
    for pid, proc in sorted(targets.items()):
        description = f"PID {pid} ({_cmdline(proc) or '?'})"
        try:
            proc.terminate()
            stopped.append(description)
        except psutil.NoSuchProcess:
            continue
        except psutil.AccessDenied:
            print(f"Cannot stop {description}: access denied")
    _, alive = psutil.wait_procs(list(targets.values()), timeout=5)
    for proc in alive:
        try:
            proc.kill()
        except psutil.NoSuchProcess:
            pass
    psutil.wait_procs(alive, timeout=5)
    for line in stopped:
        print(f"Stopped {line}")
    deadline = time.monotonic() + timeout_s
    while True:
        holders = listening_pids(port) - {me}
        if not holders:
            return stopped
        if time.monotonic() >= deadline:
            raise RuntimeError(
                f"port {port} is still held by "
                + ", ".join(_describe(pid) for pid in sorted(holders))
                + f" after {timeout_s:.0f} s; stop it by hand before launching"
            )
        time.sleep(0.2)


def start_server(argv: Sequence[str], log_path,
                 env_overrides: Optional[Mapping[str, str]] = None,
                 cwd=None) -> subprocess.Popen:
    """Start `argv` unbuffered with stdout+stderr in `log_path` (truncated)."""
    env = os.environ.copy()
    env.update(env_overrides or {})
    env["PYTHONUNBUFFERED"] = "1"
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log = open(log_path, "wb", buffering=0)
    try:
        proc = subprocess.Popen(list(argv), stdout=log, stderr=subprocess.STDOUT, env=env,
                                cwd=None if cwd is None else str(cwd))
    finally:
        log.close()
    print(f"Started PID {proc.pid}: {' '.join(argv)}")
    print(f"Log → {log_path}")
    return proc


def _probe(port: int, health_url: Optional[str], ready: Optional[Callable[[dict], bool]]):
    """Return (answered, health): answered is True once the server counts as ready."""
    if health_url:
        try:
            with urllib.request.urlopen(health_url, timeout=2) as response:
                health = json.loads(response.read().decode())
        except Exception:
            return False, None
        if not isinstance(health, dict):
            return False, None
        ok = ready(health) if ready is not None else health.get("status") == "ready"
        return bool(ok), health
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1):
            return True, None
    except OSError:
        return False, None


def _own_pids(proc) -> set:
    pids = {proc.pid}
    try:
        pids.update(child.pid for child in psutil.Process(proc.pid).children(recursive=True))
    except _ACCESS_ERRORS:
        pass
    return pids


def wait_until_ready(proc, port: int, log_path, timeout_s: float,
                     health_url: Optional[str] = None,
                     ready: Optional[Callable[[dict], bool]] = None,
                     banner: Optional[str] = None,
                     settle_s: float = 2.0, poll_s: float = 1.0):
    """Wait until the server this cell started owns `port` and is ready.

    Returns the /health dict in health mode, None in TCP mode. Raises RuntimeError on an
    early exit, a foreign/stale owner or a missing launcher banner, and TimeoutError (after
    terminating `proc`) when the deadline passes.
    """
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        code = proc.poll()
        if code is not None:
            raise RuntimeError(
                f"server PID {proc.pid} exited with code {code} before it was ready. "
                f"Last log lines of {log_path}:\n{_tail(log_path)}"
            )
        answered, health = _probe(port, health_url, ready)
        if answered:
            owners = listening_pids(port)
            mine = _own_pids(proc)
            if not owners or not owners <= mine:
                foreign = ", ".join(_describe(pid) for pid in sorted(owners - mine))
                foreign = foreign or "no visible process"
                raise RuntimeError(
                    f"port {port} answered but is owned by {foreign}, not the server this cell "
                    f"started (PID {proc.pid}). That is a stale server; re-run the launch cell, "
                    "which stops old servers first."
                )
            time.sleep(settle_s)
            code = proc.poll()
            if code is not None:
                raise RuntimeError(
                    f"server PID {proc.pid} exited with code {code} right after binding port "
                    f"{port}. Last log lines of {log_path}:\n{_tail(log_path)}"
                )
            if banner is not None:
                text = Path(log_path).read_text(errors="replace") if Path(log_path).exists() else ""
                if banner not in text:
                    raise RuntimeError(
                        f"expected launcher banner '{banner}' not found in {log_path}, so this "
                        "is not the patched launcher. Re-run the embed cells, then the launch cell."
                    )
            print(f"READY: PID {proc.pid} ({_cmdline(psutil.Process(proc.pid))}) owns port "
                  f"{port}; log → {log_path}")
            return health
        time.sleep(poll_s)
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
    raise TimeoutError(
        f"server PID {proc.pid} was not ready on port {port} within {timeout_s:.0f} s "
        f"(terminated). Last log lines of {log_path}:\n{_tail(log_path)}"
    )


def log_report(log_path, tail: int = 80, max_matches: int = 40) -> None:
    """Print the last `tail` log lines and up to `max_matches` numbered error lines."""
    path = Path(log_path)
    if not path.exists():
        print(f"{path} does not exist (has the launch cell run on this runtime?)")
        return
    lines = _read_lines(path)
    print(f"--- last {tail} lines of {path} ({len(lines)} lines total) ---")
    for line in lines[-tail:]:
        print(line)
    matches = [(number, line) for number, line in enumerate(lines, start=1)
               if ERROR_PATTERN.search(line)]
    print(f"--- {len(matches)} lines matching ERROR_PATTERN (showing up to {max_matches}) ---")
    for number, line in matches[:max_matches]:
        print(f"{number}: {line}")
    if not matches:
        print("(none)")


def process_report(port: int, expected_marker: str, proc=None) -> None:
    """Print every known server process, then check the right server owns `port`."""
    print("--- running server processes ---")
    found = False
    now = time.time()
    for candidate in psutil.process_iter():
        cmdline = _cmdline(candidate)
        if not cmdline or not any(marker in cmdline for marker in SERVER_MARKERS):
            continue
        try:
            elapsed = now - candidate.create_time()
            rss_mb = candidate.memory_info().rss / 1e6
        except _ACCESS_ERRORS:
            continue
        found = True
        print(f"PID {candidate.pid}  up {elapsed:7.0f} s  RSS {rss_mb:8.1f} MB  {cmdline}")
    if not found:
        print("(no process with a known server marker)")
    owners = listening_pids(port)
    if not owners:
        raise RuntimeError(f"nothing is listening on port {port}; run the launch cell")
    for pid in sorted(owners):
        try:
            cmdline = _cmdline(psutil.Process(pid))
        except _ACCESS_ERRORS:
            cmdline = ""
        if expected_marker not in cmdline:
            raise RuntimeError(
                f"WRONG server on port {port}: PID {pid} ({cmdline or '?'}) does not contain "
                f"'{expected_marker}'. Run the teardown cell, then the launch cell."
            )
    if proc is not None:
        mine = _own_pids(proc)
        if not owners <= mine:
            raise RuntimeError(
                f"port {port} is owned by "
                + ", ".join(_describe(pid) for pid in sorted(owners))
                + f", not by the server this notebook started (PID {proc.pid})"
            )
    print(f"OK: {expected_marker} (PID {min(owners)}) owns port {port}")


def gpu_report() -> None:
    """Print GPU memory/utilisation and the processes using the GPU."""
    queries = (
        ["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu",
         "--format=csv"],
        ["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory", "--format=csv"],
    )
    for argv in queries:
        try:
            result = subprocess.run(argv, capture_output=True, text=True, timeout=30)
        except FileNotFoundError:
            print("no nvidia-smi (CPU runtime?): pick a GPU runtime for this notebook")
            return
        print(result.stdout.rstrip() or result.stderr.rstrip())


def health_report(port: int) -> dict:
    """GET /health on the local server, pretty-print it and return it."""
    url = f"http://127.0.0.1:{port}/health"
    try:
        with urllib.request.urlopen(url, timeout=10) as response:
            health = json.loads(response.read().decode())
    except Exception as exc:
        print(f"GET {url} failed: {type(exc).__name__}: {exc}")
        raise
    print(json.dumps(health, indent=2, sort_keys=True))
    return health


def stop_server(proc, port: int, patterns: Sequence[str] = ()) -> None:
    """Teardown: stop `proc` (if alive) and any leftover server, then confirm the port is free."""
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=10)
        print(f"Stopped PID {proc.pid}")
    stop_servers(port, patterns)
    print(f"port {port} is free")


In [ ]:
import hashlib
from pathlib import Path

EXPECTED_SHA256 = '2d8f63cbfdf5e679c268dd4f4436129e8fa8f30acfd5c5f5d88eb1082c2bf2a8'
actual = hashlib.sha256(Path('/content/colab_server_utils.py').read_bytes()).hexdigest()
if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f'/content/colab_server_utils.py has sha256 {actual[:12]}, expected {EXPECTED_SHA256[:12]}: '
        're-run the %%writefile cell above. To change the file, edit control/vla_bridge/colab_server_utils.py '
        'and regenerate the notebooks (scripts/build_colab_notebooks.py).'
    )
print(f'/content/colab_server_utils.py matches control/vla_bridge/colab_server_utils.py (sha256 {actual[:12]})')

## Tier C: Octo resident adapter (Plan 14-10, `--backend octo`)

Octo has no LeRobot `PolicyServer` integration and is **JAX/Flax**, not PyTorch. It runs as its own
resident Flask adapter, `control/vla_bridge/octo_server.py`, built like the OpenVLA adapters (see README): it
loads once, warms up (JIT-compiles), then serves. It does not use the gRPC PolicyServer lane.

The code comes from **github.com/octo-models/octo**, cloned at the pinned commit `241fb35` (MIT). It is
not on PyPI. The adapter loads through Octo's own `OctoModel.load_pretrained` / `create_tasks` /
`sample_actions`, exactly as the repo's `examples/03` and `examples/04` do. It never uses Octo's
`hf://` path, which downloads `main` and every checkpoint step. Instead it downloads one step of the
pinned revision, and it pins the `t5-base` tokenizer to `google-t5/t5-base@a9723ea`.

**Status:** written from upstream source and HF Hub metadata. These cells have **not yet been run
on Colab**.

Requirements:
- **GPU:** any Colab GPU. Octo-Base is 93M parameters and Octo-Small is 27M.
- **Dependencies:** Octo pins Python 3.10, `jax[cuda11_pip]==0.4.20`, `flax==0.7.5`,
  `tensorflow==2.15.0` and `numpy==1.24.3`. None of those may touch the PolicyServer's environment, so
  the adapter runs in its own venv. `octo_server.py` refuses to start on other versions.

Default checkpoint: `sapanostic/octo-so101-3hz-finetune@a45ff970`, step 10000. It is a real **SO-101**
fine-tune of `octo-base-1.5`, trained at 3 Hz with the task "pick up the cube", on primary + wrist
cameras with a 2-frame window. Its action is a 7-D end-effector **delta**: metres, roll/pitch/yaw
differences, and an absolute gripper. The control side converts it with IK. The official MIT
`rail-berkeley/octo-{small,base}-1.5` checkpoints, with the `bridge_dataset` statistics, are reviewed
alternates.

In [ ]:
# Octo only: the official install (README: pip install -e . + requirements.txt + jax cuda11 wheel)
# in an isolated Python 3.10 venv, at a pinned octo-models/octo commit. Extra exact pins fix what
# requirements.txt leaves open (orbax-checkpoint, transformers) to versions contemporary with
# jax 0.4.20 / flax 0.7.5. Clone, install and import check are all verified by exit status.
import subprocess, sys
from pathlib import Path
OCTO_COMMIT = '241fb3514b7c40957a86d869fecb7c7fc353f540'
OCTO_DIR = '/content/octo'
OCTO_VENV = '/content/octo_venv'
OCTO_PYTHON = f'{OCTO_VENV}/bin/python'
if not Path(OCTO_DIR).exists():
    subprocess.run(['git', 'clone', 'https://github.com/octo-models/octo', OCTO_DIR], check=True)
subprocess.run(['git', '-C', OCTO_DIR, 'checkout', '--detach', OCTO_COMMIT], check=True)
head = subprocess.run(['git', '-C', OCTO_DIR, 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True)
if head.stdout.strip() != OCTO_COMMIT:
    raise RuntimeError(f'octo checkout is at {head.stdout.strip()}, expected {OCTO_COMMIT}')
if not Path(OCTO_PYTHON).exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
    subprocess.run(['uv', 'venv', '--python', '3.10', '--seed', OCTO_VENV], check=True)
install = subprocess.run(
    ['uv', 'pip', 'install', '--python', OCTO_PYTHON,
     '--find-links', 'https://storage.googleapis.com/jax-releases/jax_cuda_releases.html',
     '-r', f'{OCTO_DIR}/requirements.txt', '-e', OCTO_DIR,
     'jax[cuda11_pip]==0.4.20', 'flax==0.7.5', 'orbax-checkpoint==0.4.4', 'transformers==4.34.1',
     'numpy==1.24.3', 'flask', 'opencv-python-headless==4.11.0.86', 'pillow', 'requests', 'huggingface_hub'],
    capture_output=True, text=True)
print(install.stdout[-2000:]); print(install.stderr[-2000:])
if install.returncode != 0:
    raise RuntimeError(f'Octo venv install failed (exit {install.returncode})')
check = subprocess.run([OCTO_PYTHON, '-c', (
    "import importlib.metadata as m, jax\n"
    "print({p: m.version(p) for p in ('jax', 'jaxlib', 'flax', 'orbax-checkpoint', 'tensorflow', 'numpy', 'transformers')})\n"
    "from octo.model.octo_model import OctoModel\n"
    "assert jax.default_backend() == 'gpu', f'JAX backend is {jax.default_backend()}, expected gpu'\n"
)], capture_output=True, text=True)
print(check.stdout); print(check.stderr[-2000:])
if check.returncode != 0:
    raise RuntimeError('Octo venv import check failed; see stderr above')

## Adapter source and launch notes

The next cell writes `control/vla_bridge/octo_server.py` into `/content/octo_server.py`, and the cell after it checks its sha256. The launch cell then starts it as a separate OS process.

It runs with the venv's Python.

On first start it downloads two things:
- the pinned checkpoint revision's `config.json`, `dataset_statistics.json` and `example_batch.msgpack`,
  plus **one** step of params (about 0.8 GB for Octo-Base)
- the pinned `google-t5/t5-base` tokenizer files

`/health` only answers after the model has loaded and the JIT warm-up has run. It reports `action_space`,
`window_size`, `action_horizon`, the action mean/std/mask, and which rig camera feeds `image_primary` and
`image_wrist`. Logs go to `/content/octo_server.log`.

The default primary camera is `overhead_left`. To use another view, add
`--primary-image-key overhead_right` to the launch cell's command.

To use an official alternate, add `--checkpoint rail-berkeley/octo-small-1.5` to the launch cell's
command. Its reviewed revision, step and `bridge_dataset` key are applied automatically. Pass the
**same** `--checkpoint` to `run_vla_episode.py`, because it selects the client-side gripper mapping.

In [ ]:
%%writefile /content/octo_server.py
"""Resident Octo HTTP adapter (Tier C, Plan 14-10, VLAB-08). Run on Colab inside
the isolated Octo venv (see vla_bridge/notebooks/http_octo.ipynb).

Same shape as `openvla_server.py` / `openvla_oft_server.py` / `fastfs_server.py`:
- the model loads ONCE
- one throwaway warmup inference runs before the socket opens
- `/health` reports ready
- a non-blocking `inference_lock` answers concurrent requests with 429
- `ValueError`/`KeyError`/`TypeError` on a request become 400
- a model failure (`RuntimeError`, e.g. a non-finite action) becomes 500,
  so the client holds position instead of parsing NaN JSON

The server stays model-format-only. It returns the first step of Octo's
un-normalized action chunk, and `backends/octo.py` turns that into a
`JOINT_ORDER` target for the checkpoint's action space.

Verified against live upstream source (2026-10-01, execution discipline)
------------------------------------------------------------------------
github.com/octo-models/octo @ 241fb35 (main HEAD; tag v1.5 = 5eaa5c6 + a
setup.py packaging fix), MIT. Octo is JAX/Flax, not PyTorch, and is NOT on
PyPI (14-RESEARCH audit). Files read: `README.md`, `requirements.txt`,
`octo/model/octo_model.py`, `octo/utils/gym_wrappers.py`,
`octo/model/components/tokenizers.py`, `octo/data/utils/text_processing.py`,
`octo/data/oxe/oxe_standardization_transforms.py`,
`octo/data/utils/data_utils.py`, `examples/03_eval_finetuned.py`,
`examples/04_eval_finetuned_on_robot.py`, `examples/envs/widowx_env.py`.

- Loading: `OctoModel.load_pretrained(checkpoint_path, step)` reads
  `config.json`, `example_batch.msgpack`, `dataset_statistics.json` and the
  orbax params at `<step>/`. Its own `hf://` path calls
  `snapshot_download(repo)` with NO revision (resolves `main`) and downloads
  every step a repo ships. So `_download_pinned` fetches only the pinned
  revision's chosen step into a local dir and loads that (T-14-06).
- Text: the config's `HFTokenizer` and `LanguageTokenizer` both resolve
  `"t5-base"` from the HF Hub at `main` (tokenizer files + `AutoConfig`; the
  T5 encoder weights are inside the Octo checkpoint). `_pin_text_encoder`
  fetches `google-t5/t5-base` at a reviewed sha and rewrites the local
  `config.json` to point at it. Any other encoder name is refused.
- Inference (examples 03/04): `tasks = model.create_tasks(texts=[...])`,
  `model.sample_actions(obs, tasks, unnormalization_statistics=<stats>["action"],
  rng=...)` -> `(batch, action_horizon, action_dim)`. Observations are
  `(batch, window, ...)` with `timestep_pad_mask`. `HistoryWrapper` repeats the
  first frame and pads the older slots on reset; missing keys count as padded.
- Images (`ResizeImageWrapper`): Lanczos resize to the example batch's size,
  then for `image_primary` only, the average of the training random-resized-crop
  (scale 0.9): a centre crop of sqrt(0.9) of each side, resized back. Reproduced
  with PIL (like openvla_server.py), so the pixels differ slightly from TF's.
- Un-normalization: `NormalizationType.NORMAL` (`mean`/`std`) where `mask` is
  true; masked dims (the gripper) come back raw.
- Dependency pins (README + requirements.txt): Python 3.10, `jax[cuda11_pip]==0.4.20`,
  `flax==0.7.5`, `tensorflow==2.15.0`, `numpy==1.24.3`. `main()` refuses
  to start on other versions.

Checkpoints (HF API, read 2026-10-01; see `CHECKPOINT_PROFILES`)
----------------------------------------------------------------
- Default: `sapanostic/octo-so101-3hz-finetune` @ a45ff97, step 10000. It is
  `rail-berkeley/octo-base-1.5` fine-tuned (`encoder_and_head`, window 2, horizon 4)
  on a real SO-101 (`so_follower`) lerobot dataset,
  `sapanostic/so101_pick_cube_v2_merged`: 100 episodes, 15 Hz, `right`/`left`/`wrist`
  cameras, task "pick up the cube". It was converted to RLDS at 3 Hz
  (`ivlabs/so101_pick_cube_v2_merged_rlds_3hz`, Apache-2.0, documented card).
  Its action, confirmed from the checkpoint's own `dataset_statistics.json`
  (7-D, mask `[T]*6+[F]`, deltas of a few cm / 0.1 rad) and the dataset card, is
  `action/eef_delta_m_rad` = [dx, dy, dz (m), droll, dpitch, dyaw (rad),
  gripper_normalized]. The pose is Pinocchio FK on the SO101 URDF
  (`so101_new_calib.urdf`), and the gripper is absolute, min-max normalized
  over [-10, 100]. So it is an **EEF delta: the client runs IK**.
  `finetune_config.json`: primary = `image_1`, wrist = `image_2`.
- Alternates: the official MIT `rail-berkeley/octo-{small,base}-1.5`
  (OXE-pretrained) with the `bridge_dataset` statistics, which is what
  `examples/04` uses on the WidowX. Bridge's action is `relabel_actions`'s
  state differences: [dx, dy, dz, droll, dpitch, dyaw] in the WidowX base
  frame at 5 Hz, plus the binarized gripper (1 = open).

Known limitations (documented, not hidden)
------------------------------------------
- The default checkpoint has no model card or license, and its training code
  (`so101_dataset_transform`) is unpublished. Its action space is confirmed
  from its statistics and the dataset card, not from code.
- Which source camera `image_1` is, is inferred as `left` (the dataset's camera
  order is right, left, wrist). So the default primary is this rig's
  `overhead_left`, and `--primary-image-key` overrides it.
- Actions are 3 Hz steps. The control loop applies each one as a delta from the
  current pose, so it can run faster than trained. The safety validator's
  per-step clamp bounds that.
- The server keeps a 2-frame history (Octo's window). It resets on a new
  instruction, on `"reset": true`, or after `HISTORY_RESET_S` seconds idle.
- It returns only the first step of each chunk (receding horizon, execute 1),
  not `examples/04`'s temporal ensembling.

Colab launch: see vla_bridge/notebooks/http_octo.ipynb.
"""
import argparse
import base64
import collections
import json
import math
import os
import threading
import time
from pathlib import Path

import cv2
import numpy as np
from PIL import Image

DEFAULT_CHECKPOINT = "sapanostic/octo-so101-3hz-finetune"
DEFAULT_REVISION = "a45ff970242a66e268871a5f64530947143f9a6e"
DEFAULT_LOCAL_DIR = "/content/octo_checkpoints"
DEFAULT_PORT = 3004  # 3000 FastFS, 3001 OpenVLA-OFT, 3002 Wall-X, 3003 OpenVLA
OCTO_COMMIT = "241fb3514b7c40957a86d869fecb7c7fc353f540"  # github.com/octo-models/octo main HEAD

# T-14-06: the only tokenizer/encoder config Octo may resolve (HF API sha, read 2026-10-01).
T5_REPO = "google-t5/t5-base"
T5_REVISION = "a9723ea7f1b39c1eae772870f3b547bf6ef7e6c1"
T5_FILES = ("config.json", "spiece.model", "tokenizer.json")
REVIEWED_ENCODER = "t5-base"

# Reviewed checkpoints. `backend_configs/octo.json["checkpoints"]` must agree
# (pinned by test_client_and_server_checkpoint_profiles_agree).
CHECKPOINT_PROFILES = {
    "sapanostic/octo-so101-3hz-finetune": dict(
        revision="a45ff970242a66e268871a5f64530947143f9a6e", step=10000, unnorm_key=None,
        action_space="eef_delta"),
    "rail-berkeley/octo-small-1.5": dict(
        revision="dc9aa3019f764726c770814b27e4ab0fc6e32a58", step=300000, unnorm_key="bridge_dataset",
        action_space="eef_delta"),
    "rail-berkeley/octo-base-1.5": dict(
        revision="ee3c10e8edd6ce2e8b1e8744d3c6fba4097bed48", step=300000, unnorm_key="bridge_dataset",
        action_space="eef_delta"),
}

CENTER_CROP_SCALE = 0.9  # ResizeImageWrapper avg_scale
MAX_IMAGE_SIDE = 1024
IMAGE_KEYS = ("overhead_left", "overhead_right", "wrist")
HISTORY_RESET_S = 10.0
PINNED_DEPENDENCIES = {"jax": {"0.4.20"}, "flax": {"0.7.5"}, "tensorflow": {"2.15.0"}, "numpy": {"1.24.3"}}


def _download_pinned(checkpoint, revision, local_dir, step):
    """Materialise the pinned revision's metadata plus ONE step of params as a
    real directory (Octo's own `hf://` path resolves `main` and fetches every step)."""
    from huggingface_hub import snapshot_download

    target = Path(local_dir) / checkpoint.replace("/", "--") / revision
    patterns = ["config.json", "dataset_statistics.json", "example_batch.msgpack", f"{step}/*"]
    return snapshot_download(repo_id=checkpoint, revision=revision, local_dir=str(target),
                             allow_patterns=patterns)


def _pin_text_encoder(model_dir, local_dir):
    """Point the checkpoint's `t5-base` tokenizer and encoder config at a local
    copy of the reviewed `google-t5/t5-base` revision. Refuses any other encoder."""
    from huggingface_hub import snapshot_download

    t5_dir = Path(local_dir) / "google-t5--t5-base" / T5_REVISION
    path = Path(model_dir) / "config.json"
    config = json.loads(path.read_text())
    local = str(t5_dir)
    refs = []
    if config.get("text_processor"):
        refs.append((config["text_processor"]["kwargs"], "tokenizer_name"))
    for spec in (config["model"].get("task_tokenizers") or {}).values():
        kwargs = spec.get("kwargs") or {}
        if kwargs.get("encoder") is not None:
            refs.append((kwargs, "encoder"))
    for holder, key in refs:
        if holder[key] not in (REVIEWED_ENCODER, local):
            raise RuntimeError(f"{path}: {key}={holder[key]!r} is not the reviewed {REVIEWED_ENCODER!r}")
    snapshot_download(repo_id=T5_REPO, revision=T5_REVISION, local_dir=local, allow_patterns=list(T5_FILES))
    for holder, key in refs:
        holder[key] = local  # contains "t5": LanguageTokenizer then builds FlaxT5EncoderModel
    path.write_text(json.dumps(config) + "\n")


def _load_policy(model_dir, step):
    """The official load (README / examples 03-04)."""
    import tensorflow as tf

    tf.config.set_visible_devices([], "GPU")  # Octo imports TF for I/O only; JAX owns the GPU
    from octo.model.octo_model import OctoModel as UpstreamOctoModel

    return UpstreamOctoModel.load_pretrained(model_dir, step)


def _sample_actions(model, observations, tasks, unnormalization_statistics, seed):
    """`model.sample_actions(...)` with a fresh PRNG key, as examples 03/04 call it
    (default `NormalizationType.NORMAL`, the reviewed checkpoints' type); returns numpy."""
    import jax

    actions = model.sample_actions(observations, tasks, unnormalization_statistics=unnormalization_statistics,
                                   rng=jax.random.PRNGKey(seed))
    return np.asarray(actions)


def preprocess_image(rgb, size, center_crop):
    """HxWx3 uint8 RGB -> `size` (h, w) uint8 RGB: Lanczos resize, then
    optionally ResizeImageWrapper's average crop (sqrt(0.9) per side)."""
    height, width = size
    image = Image.fromarray(np.ascontiguousarray(rgb)).convert("RGB")
    image = image.resize((width, height), Image.LANCZOS)
    if center_crop:
        side = math.sqrt(CENTER_CROP_SCALE)
        dx, dy = width * (1 - side) / 2, height * (1 - side) / 2
        image = image.resize((width, height), Image.BILINEAR, box=(dx, dy, width - dx, height - dy))
    return np.asarray(image, dtype=np.uint8)


def _resolve_action_stats(statistics, unnorm_key, checkpoint):
    if unnorm_key is not None:
        if unnorm_key not in statistics or not isinstance(statistics[unnorm_key], dict):
            raise ValueError(f"unnorm key {unnorm_key!r} not in {checkpoint}'s dataset_statistics "
                             f"{sorted(statistics)}")
        return unnorm_key, statistics[unnorm_key]["action"]
    if isinstance(statistics.get("action"), dict) and "mean" in statistics["action"]:
        return None, statistics["action"]  # a single-dataset fine-tune
    keys = sorted(statistics)
    if len(keys) != 1:
        raise ValueError(f"{checkpoint} has unnorm keys {keys}; pass --unnorm-key to choose one")
    return keys[0], statistics[keys[0]]["action"]


class OctoModel:
    def __init__(self, checkpoint=DEFAULT_CHECKPOINT, revision=None, step=None, unnorm_key=None,
                 local_dir=DEFAULT_LOCAL_DIR, primary_image_key="overhead_left", wrist_image_key="wrist",
                 center_crop=True, seed=0):
        for key in (primary_image_key, wrist_image_key):
            if key not in IMAGE_KEYS:
                raise ValueError(f"image keys must be among {IMAGE_KEYS}")
        profile = CHECKPOINT_PROFILES.get(checkpoint, {})
        if step is None:
            step = profile.get("step")
        if revision:
            model_dir = _download_pinned(checkpoint, revision, local_dir, step)
            _pin_text_encoder(model_dir, local_dir)
        else:
            model_dir = f"hf://{checkpoint}"  # explicit opt-out: Octo's own unpinned path
            step = None  # load_pretrained refuses a step for hf:// paths
        self.model = _load_policy(model_dir, step)

        observation = self.model.example_batch["observation"]
        if "proprio" in observation:
            raise RuntimeError(f"{checkpoint} expects a proprio observation, which this adapter does not "
                               "supply (it would be the checkpoint's own FK pose); not supported")
        if "image_primary" not in observation:
            raise RuntimeError(f"{checkpoint} has no image_primary observation")
        self.window = int(observation["timestep_pad_mask"].shape[1])
        self.image_sizes = {k: tuple(int(v) for v in observation[k].shape[2:4])
                            for k in ("image_primary", "image_wrist") if k in observation}
        self.image_sources = {"image_primary": primary_image_key, "image_wrist": wrist_image_key}
        self.image_sources = {k: v for k, v in self.image_sources.items() if k in self.image_sizes}
        self.required_image_keys = tuple(dict.fromkeys(self.image_sources.values()))

        if unnorm_key is None:
            unnorm_key = profile.get("unnorm_key")
        self.unnorm_key, self.action_stats = _resolve_action_stats(
            self.model.dataset_statistics, unnorm_key, checkpoint)
        self.action_dim = len(self.action_stats["mean"])
        action = self.model.example_batch.get("action")
        self.action_horizon = int(action.shape[-2]) if action is not None else None
        self.center_crop = center_crop
        self._seed = int(seed)
        self._history = collections.deque(maxlen=self.window)
        self._num_obs = 0
        self._last_instruction = None
        self._last_time = None
        self._tasks = {}
        self.model_version = f"{checkpoint}@{revision[:12] if revision else 'unpinned'}"
        stats = {k: np.asarray(v).tolist() for k, v in self.action_stats.items()}
        self.metadata = dict(
            checkpoint=checkpoint, revision=revision or "unpinned", step=step,
            octo_commit=OCTO_COMMIT, text_encoder=f"{T5_REPO}@{T5_REVISION}" if revision else "unpinned",
            unnorm_key=self.unnorm_key, action_space=profile.get("action_space", "unknown"),
            action_dim=self.action_dim, action_horizon=self.action_horizon, window_size=self.window,
            action_mean=stats["mean"], action_std=stats["std"],
            action_mask=stats.get("mask", [True] * self.action_dim),
            normalization_type="normal", image_keys=self.image_sources,
            image_sizes={k: list(v) for k, v in self.image_sizes.items()}, center_crop=center_crop,
            returns="first step of the un-normalized action chunk",
        )

    def reset(self):
        self._history.clear()
        self._num_obs = 0
        self._last_instruction = None

    def _observation(self, images, instruction, reset):
        now = time.monotonic()
        stale = self._last_time is not None and now - self._last_time > HISTORY_RESET_S
        if reset or stale or instruction != self._last_instruction:
            self.reset()
        self._last_time = now
        self._last_instruction = instruction
        frame = {key: preprocess_image(images[source], self.image_sizes[key],
                                       self.center_crop and key == "image_primary")
                 for key, source in self.image_sources.items()}
        if self._num_obs == 0:
            self._history.extend([frame] * self.window)  # HistoryWrapper.reset
        else:
            self._history.append(frame)
        self._num_obs += 1
        observation = {key: np.stack([f[key] for f in self._history])[None] for key in frame}
        pad = np.ones((1, self.window), dtype=bool)
        pad[0, : self.window - min(self._num_obs, self.window)] = False
        observation["timestep_pad_mask"] = pad
        return observation

    def __call__(self, images, instruction, reset=False):
        observation = self._observation(images, instruction, reset)
        if instruction not in self._tasks:
            self._tasks[instruction] = self.model.create_tasks(texts=[instruction])
        self._seed += 1
        chunk = np.asarray(_sample_actions(self.model, observation, self._tasks[instruction], self.action_stats,
                                           self._seed), dtype=np.float64)
        if chunk.ndim != 3 or chunk.shape[0] != 1 or chunk.shape[2] != self.action_dim:
            raise RuntimeError(f"model returned actions of shape {chunk.shape}, "
                               f"expected (1, horizon, {self.action_dim})")
        values = chunk[0, 0]
        if not np.all(np.isfinite(values)):
            raise RuntimeError(f"model returned a non-finite action {values.tolist()}")
        return [float(v) for v in values]


def _decode_rgb(b64_png):
    """base64 PNG (OpenCV BGR-encoded by the client) -> HxWx3 uint8 RGB."""
    if not isinstance(b64_png, str):
        raise TypeError("image must be a base64 string")
    raw = base64.b64decode(b64_png, validate=True)
    frame = cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), cv2.IMREAD_COLOR)
    if frame is None:
        raise ValueError("invalid PNG")
    if max(frame.shape[:2]) > MAX_IMAGE_SIDE:
        raise ValueError(f"image dimensions must be <= {MAX_IMAGE_SIDE}")
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def create_app(infer):
    from flask import Flask, jsonify, request
    app = Flask(__name__)
    app.config['MAX_CONTENT_LENGTH'] = 20 * 1024 * 1024
    inference_lock = threading.Lock()
    image_keys = tuple(getattr(infer, 'required_image_keys', ('overhead_left', 'wrist')))

    @app.get('/health')
    def health():
        return jsonify(status='ready', protocol_version=1, model=infer.metadata)

    @app.post('/action')
    def action():
        if not inference_lock.acquire(blocking=False):
            return jsonify(error='octo service busy'), 429
        try:
            body = request.get_json(silent=True)
            if not isinstance(body, dict):
                raise ValueError('expected a JSON object body')
            images_in = body['images']
            if not isinstance(images_in, dict):
                raise TypeError('images must be an object')
            images = {key: _decode_rgb(images_in[key]) for key in image_keys}
            instruction = body['instruction']
            if not isinstance(instruction, str) or not instruction.strip():
                raise ValueError('instruction must be a non-empty string')
            joint_state = body.get('joint_state')
            if joint_state is not None and not isinstance(joint_state, dict):
                raise TypeError('joint_state must be an object')
            reset = body.get('reset', False)
            if not isinstance(reset, bool):
                raise TypeError('reset must be a boolean')
            if reset and hasattr(infer, 'reset'):
                infer.reset()
            started = time.perf_counter()
            result = infer(images, instruction)
            elapsed = (time.perf_counter() - started) * 1000
            return jsonify(action=[float(v) for v in result], model_version=infer.model_version,
                           inference_ms=elapsed)
        except (ValueError, KeyError, TypeError) as exc:
            return jsonify(error=str(exc)), 400
        except RuntimeError as exc:
            return jsonify(error=f'octo inference failed: {exc}'), 500
        finally:
            inference_lock.release()
    return app


def _installed_versions():
    import importlib.metadata as metadata

    return {name: metadata.version(name) for name in PINNED_DEPENDENCIES}


def _check_dependency_versions():
    versions = _installed_versions()
    bad = {name: version for name, version in versions.items() if version not in PINNED_DEPENDENCIES[name]}
    if bad:
        raise SystemExit(
            f"Octo needs jax==0.4.20, flax==0.7.5, tensorflow==2.15.0, numpy==1.24.3 "
            f"(octo README / requirements.txt); found {bad}. Use the isolated Octo venv."
        )


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--checkpoint', default=DEFAULT_CHECKPOINT)
    parser.add_argument('--revision', default=None,
                        help="pinned HF commit sha (defaults to the reviewed profile's); "
                             "pass '' only to deliberately load the unpinned repo id")
    parser.add_argument('--step', type=int, default=None, help="checkpoint step (defaults to the profile's)")
    parser.add_argument('--unnorm-key', default=None)
    parser.add_argument('--local-dir', default=DEFAULT_LOCAL_DIR)
    parser.add_argument('--primary-image-key', default='overhead_left', choices=IMAGE_KEYS)
    parser.add_argument('--wrist-image-key', default='wrist', choices=IMAGE_KEYS)
    parser.add_argument('--no-center-crop', action='store_true')
    parser.add_argument('--seed', type=int, default=0)
    parser.add_argument('--port', type=int, default=DEFAULT_PORT)
    args = parser.parse_args()
    profile = CHECKPOINT_PROFILES.get(args.checkpoint, {})
    revision = args.revision if args.revision is not None else profile.get('revision')
    if revision is None:
        raise SystemExit(f"{args.checkpoint} has no reviewed profile; pass --revision <sha> "
                         "(or --revision '' to deliberately load it unpinned)")
    step = args.step if args.step is not None else profile.get('step')
    if revision and step is None:
        raise SystemExit(f"{args.checkpoint} has no reviewed profile; pass --step <n> "
                         "(a pinned download fetches exactly one checkpoint step)")
    _check_dependency_versions()
    # JAX shares the GPU with nothing else in this process, but keep it from
    # grabbing 75 % of VRAM up front (FastFS may run on the same runtime).
    os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
    model = OctoModel(args.checkpoint, revision=revision or None, step=step, unnorm_key=args.unnorm_key,
                      local_dir=args.local_dir, primary_image_key=args.primary_image_key,
                      wrist_image_key=args.wrist_image_key, center_crop=not args.no_center_crop,
                      seed=args.seed)
    # JIT compilation happens before the socket opens and readiness is reported.
    model({key: np.zeros((256, 256, 3), dtype=np.uint8) for key in model.required_image_keys}, 'warm up')
    model.reset()  # the warmup frame must not become the first real request's history
    print(f'Octo model loaded and warmed ({model.model_version}); ready', flush=True)
    create_app(model).run(host='0.0.0.0', port=args.port, threaded=True, use_reloader=False)


if __name__ == '__main__':
    main()


In [ ]:
import hashlib
from pathlib import Path

EXPECTED_SHA256 = '0c1abb5004c8065693039b7e6c3dcbbff9d4849bbd1729a443eb8d6954b858f7'
actual = hashlib.sha256(Path('/content/octo_server.py').read_bytes()).hexdigest()
if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f'/content/octo_server.py has sha256 {actual[:12]}, expected {EXPECTED_SHA256[:12]}: '
        're-run the %%writefile cell above. To change the file, edit control/vla_bridge/octo_server.py '
        'and regenerate the notebooks (scripts/build_colab_notebooks.py).'
    )
print(f'/content/octo_server.py matches control/vla_bridge/octo_server.py (sha256 {actual[:12]})')

## Launch

The launch cell first stops any earlier server: every process whose command line contains one of `octo_server.py`, and whatever listens on port 3004.
It then starts this notebook's server as a separate OS process (unbuffered, log at `/content/octo_server.log`).
It only succeeds once **its own** still-running process owns port 3004.
A stale server answering on the port is an error, never a successful launch (in the 2026-10-18 live session a stale stock server made a diffusion launch look successful).

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import start_server, stop_servers, wait_until_ready

PORT = 3004
OCTO_LOG = '/content/octo_server.log'
STOP_PATTERNS = ['octo_server.py']
OCTO_PYTHON = '/content/octo_venv/bin/python'

stop_servers(PORT, STOP_PATTERNS)
octo_server_process = start_server(
    [OCTO_PYTHON, '-u', '/content/octo_server.py', '--port', '3004'],
    OCTO_LOG,
    env_overrides={'XLA_PYTHON_CLIENT_PREALLOCATE': 'false', 'TOKENIZERS_PARALLELISM': 'false'},
)
health = wait_until_ready(
    octo_server_process, PORT, OCTO_LOG,
    timeout_s=1200, health_url='http://127.0.0.1:3004/health',
)
print('Octo adapter ready:', {k: health['model'][k] for k in (
    'checkpoint', 'revision', 'step', 'action_space', 'window_size', 'action_horizon', 'image_keys')})

## Open an HTTP tunnel to port 3004

In [ ]:
from pyngrok import ngrok

octo_tunnel = ngrok.connect(3004, "http")  # plain JSON-over-HTTP
print(f"Octo endpoint: {octo_tunnel.public_url}")
SMOKE = ".venv/bin/python -m vla_bridge.backend_smoke --backend octo --server-address https://<octo_tunnel> --instruction 'pick up the cube'"
EPISODE = '.venv/bin/python run_vla_episode.py --backend octo --server-address https://OCTO_TUNNEL.ngrok-free.app --instruction "pick up the cube" --out outputs/octo_UNIQUE_NAME --max-steps 20'
for placeholder in ('https://<octo_tunnel>', 'https://OCTO_TUNNEL.ngrok-free.app'):
    SMOKE = SMOKE.replace(placeholder, octo_tunnel.public_url)
    EPISODE = EPISODE.replace(placeholder, octo_tunnel.public_url)
print('Local smoke (run in control/):')
print(SMOKE)
print('Local episode (run in control/):')
print(EPISODE)

## Run locally

Then, locally in `control/` (pass the base URL; the client appends `/action`):

```bash
.venv/bin/python run_vla_episode.py --backend octo \
  --server-address https://OCTO_TUNNEL.ngrok-free.app \
  --instruction "pick up the cube" \
  --out outputs/octo_UNIQUE_NAME --max-steps 20
```

`--instruction` matches the default checkpoint's only training task. Each step's `model_version` should
read `sapanostic/octo-so101-3hz-finetune@a45ff970242a`. Each delta-EEF action goes through
`eef_to_joint.solve_ik()` before the safety validator.

For an official alternate, add `--checkpoint rail-berkeley/octo-small-1.5`, the same value the server
was launched with.

A step that logs `http-error-holding-position` means the adapter returned an error. Check
`/content/octo_server.log`:
- 500 `octo inference failed` means the model produced a non-finite or wrong-shaped action.
- 400 means a bad request.

The default checkpoint was trained at 3 Hz. Each action is about a third of a second of motion, applied
as a delta from the current pose, so the per-step safety clamp may trim it at higher loop rates.

Smoke run first (synthetic hardware, the arm cannot move; the tunnel cell prints it with the URL filled in):

```bash
.venv/bin/python -m vla_bridge.backend_smoke --backend octo --server-address https://<octo_tunnel> --instruction 'pick up the cube'
```

## Diagnostics

Run any of these after the launch cell, also from a restarted kernel: each cell re-declares
what it needs.
- Log: the tail of `/content/octo_server.log`, then every line matching `ERROR`, `Error`, `Traceback`,
  `Exception`, `OutOfMemory`, `out of memory`, `Killed` or `beginAllocateToPool`.
- Processes: every known server process on this runtime (stale servers from other notebooks
  show up here); fails unless `octo_server.py` owns port 3004.
- Package versions, GPU memory, and `/health`.

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import log_report

log_report('/content/octo_server.log')

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import process_report

process_report(3004, 'octo_server.py', globals().get('octo_server_process'))

In [ ]:
import subprocess

OCTO_PYTHON = '/content/octo_venv/bin/python'
probe = (
    "import importlib.metadata as m\n"
    "for p in ('jax', 'jaxlib', 'flax', 'orbax-checkpoint', 'tensorflow', 'numpy', 'transformers'):\n"
    "    try:\n"
    "        print(p, m.version(p))\n"
    "    except m.PackageNotFoundError:\n"
    "        print(p, 'not installed')\n"
)
result = subprocess.run([OCTO_PYTHON, '-c', probe], capture_output=True, text=True)
print(result.stdout)
print(result.stderr[-2000:])

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import gpu_report

gpu_report()

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import health_report

health_report(3004)

## Teardown (every session, no exceptions)

An unattended open tunnel into a Colab-hosted server is an
elevation-of-privilege risk (T-11-08 in this plan's threat register) —
stop both processes at the end of every session, not just when finished
for the day: the cell below disconnects this notebook's tunnel, stops the
ngrok agent, stops the server and confirms port 3004 is free. Run it only
after the local episode or smoke run has finished.

In [ ]:
# Octo teardown -- every session, after the local run has finished.
import sys

sys.path.insert(0, '/content')
from colab_server_utils import stop_server
from pyngrok import ngrok

if 'octo_tunnel' in globals():
    try:
        ngrok.disconnect(octo_tunnel.public_url)
        print(f'Disconnected {octo_tunnel.public_url}')
    except Exception as exc:
        print(f'Tunnel already closed ({type(exc).__name__}: {exc})')
ngrok.kill()
print('ngrok agent stopped.')
stop_server(globals().get('octo_server_process'), 3004, ['octo_server.py'])